# [실습] 검색 에이전트: 기본 RAG에서 에이전틱 검색으로

전통적인 RAG는 질문을 한 번 검색하고 그 결과로 답을 생성하는 방식입니다.    

이는 관련 정보를 전달하여 LLM의 환각을 완화하지만, 정답이 여러 문서에 흩어져 있거나,    
특정 토큰이 포함되는 전체 내용을 찾아야 하는 질문에서는 맥락의 한계가 존재합니다.   

이번 실습에서는 기존 RAG와 Agent의 Retrieval을 비교해 보겠습니다.   

1. 일반적인 RAG의 성능이 하락하는 시나리오를 확인합니다.
2. 검색 기능을 에이전트의 도구로 연결하여, 검색 에이전트를 구성합니다.

## 1. 환경 설정

In [ ]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-qdrant qdrant-client langfuse -q

In [ ]:
import os
import re
import zipfile
from glob import glob

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.documents.base import Document
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langfuse.langchain import CallbackHandler

from select_model import load_model

load_dotenv('.env', override=True)

callbacks = [CallbackHandler()] if os.environ.get('LANGFUSE_PUBLIC_KEY') else []

PLATFORM = 'openai'
llm = load_model('gpt-5.2', platform=PLATFORM)

print('환경 설정 완료')

## 2. 문서 준비 및 벡터 스토어 구성

실습 시트의 RAG_data.zip을 다운로드하여, 저장해 주세요.

랭체인의 `TextLoader`를 이용해, 해당 문서를 불러오도록 하겠습니다.

In [ ]:
# --- 1) 압축 해제  ---
zip_path = "RAG_data.zip"
extract_path = "data"

if not os.path.isdir("data/markdowns"):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print(f"압축 해제 완료: {zip_path}")
else:
    print("data/markdowns/ 이미 존재, 압축 해제 생략")

# --- 2) 마크다운 파일 로드 ---
md_files = sorted(glob("data/markdowns/*.md"))
print(f"\n로드할 문서 {len(md_files)}개:")
for f in md_files:
    print(f"  - {os.path.basename(f)}")

documents = []
for md_file in md_files:
    loader = TextLoader(md_file, encoding="utf-8")
    docs = loader.load()
    documents.extend(docs)
print(f"\n총 {len(documents)}개 문서 로드 완료")

불러온 문서는, 전처리 후 청킹하여 DB에 저장합니다.

임베딩 모델은 text-embedding-3-large, 벡터 DB는 Qdrant를 사용합니다.

In [ ]:
# --- 3) 전처리 (HTML 엔티티, 폭 없는 공백, 연속 공백 정규화) ---
def preprocess(docs):
    for doc in docs:
        text = doc.page_content
        text = re.sub(r'&[a-zA-Z0-9#]+;', '', text)        # HTML 엔티티 제거
        for ch in (chr(0x200b), chr(0x00a0)):              # 폭 없는 공백, 줄바꿈 없는 공백 제거
            text = text.replace(ch, '')
        text = re.sub(r' {2,}', ' ', text)                 # 연속 공백 정리
        doc.page_content = text.strip()
    return docs

documents = preprocess(documents)
print("전처리 완료")

# --- 4) 청킹 ---
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"청킹 완료: {len(chunks)}개 청크 생성")

# --- 5) Qdrant 벡터 스토어 구성 ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

import uuid
uuidstr = str(uuid.uuid4())[:6]
qdrant_path = f"outputs/vectordb/search_qdrant_{uuidstr}"
# uuid: 셀 중복 실행 시 DB 충돌 방지

client = QdrantClient(path=qdrant_path)
collection_name = "ai_reports"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
    distance=Distance.COSINE,
)

vector_store.add_documents(chunks)
print(f"\nQdrant 벡터 스토어 생성 완료 ({len(chunks)}개 청크 인덱싱)")


질문 임베딩과의 코사인 유사도를 기준으로 상위 5개 청크를 선택하는 검색기(Retriever)를 구성합니다.

In [ ]:
# --- 6) Retriever 구성 ---
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
print("Retriever 구성 완료 (Top-5)")

Retriever도 invoke()를 통해 실행합니다.

In [ ]:
# Retriever 동작 확인
test_results = retriever.invoke("AI Agent가 산업에 미치는 영향은?")
for i, doc in enumerate(test_results):
    source = os.path.basename(doc.metadata.get("source", "unknown"))
    print(f'[{i+1}] {source}: {doc.page_content[:80].strip()}...')

---
## 3. 검색 도구

에이전트가 사용할 검색 도구를 두 가지로 정의합니다.

| 도구 | 검색 방식 | 사용하는 경우 |
| --- | --- | --- |
| `semantic_search` | 임베딩 유사도 기반 벡터 검색 | 표현이 달라도 주제가 비슷한 내용을 폭넓게 찾을 때 |
| `keyword_search` | 정규식 문자열 일치 | 모델명, 법안 번호처럼 정확한 토큰의 등장(Occurence)를 찾아야 할 때 |


In [ ]:
def format_docs(docs):
    return '\n\n---\n\n'.join(
        f"[출처: {os.path.basename(d.metadata.get('source', 'unknown'))}]\n{d.page_content.strip()}"
        for d in docs)


@tool
def semantic_search(query: str, k: int = 5) -> str:
    """질문과 의미가 가까운 문서 청크를 벡터 검색으로 검색합니다.
    표현이 달라도 주제가 비슷한 내용을 폭넓게 찾을 때 사용합니다."""
    docs = vector_store.as_retriever(search_kwargs={'k': k}).invoke(query)
    return format_docs(docs) if docs else '검색 결과 없음'


@tool
def keyword_search(pattern: str, max_results: int = 8) -> str:
    """정규식 패턴과 일치하는 청크를 대소문자 구분 없이 검색합니다.
    모델명, 법안 번호, 코드처럼 정확한 토큰의 일치가 필요할 때 사용합니다."""
    hits = [d for d in chunks if re.search(pattern, d.page_content, re.IGNORECASE)]
    if not hits:
        return f"'{pattern}' 일치 없음"
    head = f'총 {len(hits)}개 청크 일치 (상위 {min(len(hits), max_results)}개 표시)'
    return head + '\n\n' + format_docs(hits[:max_results])


print('검색 도구 정의 완료')

---
## 4. 기본 RAG 베이스라인

질문을 그대로 검색하고 이를 활용해 답을 생성하는 함수를 정의합니다.    

검색은 상위 k개 청크를 가져옵니다.

반환값에 사용한 출처를 함께 담아, 어떤 문서에서 답을 만들었는지 확인합니다.

In [ ]:
def single_shot_rag(question: str, k: int = 5):
    """top-k 청크를 한 번 검색해 답을 생성하고, 사용한 출처를 함께 반환합니다."""
    docs = vector_store.as_retriever(search_kwargs={'k': k}).invoke(question)
    prompt = f"다음 컨텍스트만 근거로 답하세요.\n\n{format_docs(docs)}\n\n질문: {question}"
    answer = llm.invoke(prompt).text
    sources = [os.path.basename(d.metadata.get('source', 'unknown')) for d in docs]
    return answer, sources


print('단발 RAG 베이스라인 정의 완료')

---
## 5. 회수 한계 확인: 커버리지

정답이 여러 문서에 흩어진 질문을 입력합니다.   

정답에는 SB-243 법안이 포함되어야 합니다.

In [ ]:
q_coverage = 'AI 규제와 법제도 동향을 미국, EU, 한국으로 나눠 정리해줘.'

answer_base, sources_base = single_shot_rag(q_coverage)

print(f'사용한 출처(고유 {len(set(sources_base))}개): {sorted(set(sources_base))}')
print(f"SB-243 언급: {'SB-243' in answer_base}")
print('---')
print(answer_base)

---
## 6. 단일 Retrieval 한계 확인: 전수 검색

특정 키워드에 대한 넓은 검색이 필요한 질문을 입력합니다. 

In [ ]:
ap2_total = len([d for d in chunks if 'AP2' in d.page_content])

q_exhaustive = "에이전트 결제 프로토콜 'AP2'를 언급한 내용을 모두 찾아 정리해줘."
answer_base2, sources_base2 = single_shot_rag(q_exhaustive)
recalled = sum('AP2' in d.page_content
               for d in vector_store.as_retriever(search_kwargs={'k': 5}).invoke(q_exhaustive))

print(f'코퍼스 전체 AP2 청크 수: {ap2_total}')
print(f'단발 top-5가 회수한 AP2 청크 수: {recalled}')
print(f'사용한 출처: {sorted(set(sources_base2))}')

---
## 7. 검색 에이전트

앞에서 확인한 것처럼, 1회의 검색만으로는 검색의 실패나 미흡한 점에 대해 대응하기가 어렵습니다.   

이를 해결하기 위해, Agent의 도구로 검색 기능을 재구성하겠습니다.

In [ ]:
search_agent = create_agent(
    llm,
    tools=[semantic_search, keyword_search],
    system_prompt='''당신은 문서 검색 전문가입니다. 주어진 도구로 코퍼스를 검색해 질문에 답합니다.
정답이 여러 문서에 흩어져 있을 수 있으니, 한 번의 검색에 멈추지 말고 관점을 바꿔 여러 번 검색하세요.
의미 기반 탐색은 semantic_search, 정확한 토큰의 전수 회수는 keyword_search를 사용하세요.
근거가 된 문서의 출처를 답변에 포함하고, 한국어로 답하세요.''',
)


def run_search_agent(question: str):
    """에이전트를 실행하고 답변과 도구 호출 경로를 반환합니다."""
    result = search_agent.invoke(
        {'messages': [HumanMessage(question)]},
        config={'callbacks': callbacks},
    )
    path = [tc['name'] for m in result['messages']
            for tc in (getattr(m, 'tool_calls', []) or [])]
    return result['messages'][-1].text, path


print('검색 에이전트 구성 완료')

---
## 8. 에이전트 실행과 비교

기본 RAG에서 사용한 두 질문을 에이전트로 다시 실행합니다. 도구 호출 경로와 검색 결과를 비교합니다.

In [ ]:
from IPython.display import Markdown, display

# 커버리지 질문을 에이전트로 실행
answer_agent, path_agent = run_search_agent(q_coverage)

print(f'도구 호출 경로: {path_agent}')
print(f"SB-243 언급: 단발={'SB-243' in answer_base} / 에이전트={'SB-243' in answer_agent}")
display(Markdown(answer_agent))

In [ ]:
# 전수 검색 질문을 에이전트로 실행
answer_agent2, path_agent2 = run_search_agent(q_exhaustive)

print(f'도구 호출 경로: {path_agent2}')
print(f'단발 top-5 회수: {recalled}/{ap2_total} 청크')
print(f"키워드 검색 사용: {'keyword_search' in path_agent2}")
display(Markdown(answer_agent2))

---
## 9. [실습] 검색 에이전트 확장

다음 요소를 추가해 검색 에이전트를 확장하세요.

- 특정 문서 안에서만 검색하는 도구를 정의하세요. 파일명 일부와 패턴을 받아 해당 문서의 청크만 반환합니다.
- 에이전트가 검색 횟수를 보고하도록, 도구 호출 경로를 답변과 함께 출력하세요.

---
## 10. build_agent로 검색 에이전트 배포

`build_agent`는 모델과 도구를 받아 에이전트를 만듭니다. `extra_tools`에 검색 도구를 넘기면 표준 도구와 함께 사용하는 배포용 에이전트가 만들어집니다.

`mcp_servers={}`로 MCP 연결 없이 이 노트북에서 바로 실행합니다.

In [ ]:
from build_agent import build_agent

agent = await build_agent(
    model=llm,
    extra_tools=[semantic_search, keyword_search],
    mcp_servers={},
)

result = await agent.ainvoke(
    {'messages': [{'role': 'user', 'content': "에이전트 결제 프로토콜 'AP2'를 언급한 문서를 모두 찾아줘."}]},
    config={'callbacks': callbacks},
)
print(result['messages'][-1].text)

---
## 11. 정리

이번 실습에서 구현한 내용입니다.

1. 단발 RAG 베이스라인: top-k 한 번의 검색으로 답을 생성하고 사용한 출처를 확인
2. 회수 한계 확인: 정답이 여러 문서에 흩어진 커버리지 질문과 정확 토큰의 전수 회수 질문에서 단발이 놓치는 부분을 측정
3. 검색 도구: 의미 기반 `semantic_search`와 정규식 `keyword_search`의 서로 다른 회수 특성
4. 검색 에이전트: 검색 횟수와 도구 선택을 스스로 정해 반복 검색으로 회수 범위를 확장
5. build_agent 배포: 검색 도구를 `extra_tools`로 넘겨 배포용 에이전트로 구성